# P-026 Ensemble Stacking

**Decision question:** Does ensemble stacking (ExtraTrees + RandomForest + GradientBoosting with Ridge meta-learner) improve tau-b and panel consensus over single ExtraTrees?

**Approach:**
- Base estimators: ExtraTrees (locked P-011 config), GradientBoosting (P-011 config), RandomForest (P-011 config)
- Meta-learner: Ridge regression (simple, avoids overfitting)
- StackingRegressor with cv=5 for out-of-fold predictions
- Compare across 10 random splits (seeds 0-9)
- Panel devices: 850, 1320, 119

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

df = pd.read_csv("perovskite_stability_clean.csv").fillna(0)

FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature", "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]
TARGET = "Stability_PCE_T80"

X = df[FEATURES].values
y = np.log1p(df[TARGET].values)

PANEL_IDX = [850, 1320, 119]

print(f"Dataset: {X.shape[0]} devices, {X.shape[1]} features")
print(f"Target: log1p(Stability_PCE_T80)")
print(f"Panel devices: {PANEL_IDX}")

Dataset: 1543 devices, 16 features
Target: log1p(Stability_PCE_T80)
Panel devices: [850, 1320, 119]


In [2]:
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split


def make_et():
    return ExtraTreesRegressor(
        n_estimators=700, max_features="sqrt",
        min_samples_split=3, min_samples_leaf=1,
        bootstrap=False, random_state=42, n_jobs=-1,
    )

def make_gb():
    return GradientBoostingRegressor(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        random_state=42,
    )

def make_rf():
    return RandomForestRegressor(
        n_estimators=700, max_features="sqrt",
        random_state=42, n_jobs=-1,
    )

def make_stacker():
    return StackingRegressor(
        estimators=[
            ("et", make_et()),
            ("gb", make_gb()),
            ("rf", make_rf()),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=5,
        n_jobs=-1,
    )

print("Models defined: ET (locked), GB (P-011), RF (P-011), Stacker (Ridge meta)")

Models defined: ET (locked), GB (P-011), RF (P-011), Stacker (Ridge meta)


In [3]:
import time

def panel_top20_rate(y_true, y_pred, panel_idx, test_idx):
    """For each panel device in test set, check if it lands in top-20% of predictions."""
    n = len(y_true)
    top20_cutoff = int(np.ceil(n * 0.20))
    rank_order = np.argsort(y_pred)[::-1]  # highest predicted first
    top20_set = set(rank_order[:top20_cutoff])
    hits = 0
    total = 0
    for pi in panel_idx:
        # find position of panel device in test_idx
        positions = np.where(test_idx == pi)[0]
        if len(positions) > 0:
            total += 1
            if positions[0] in top20_set:
                hits += 1
    return hits, total

results = []

for seed in range(10):
    t0 = time.time()
    train_i, test_i = train_test_split(
        np.arange(len(y)), test_size=0.20, random_state=seed
    )
    X_train, X_test = X[train_i], X[test_i]
    y_train, y_test = y[train_i], y[test_i]

    # --- ET alone ---
    et = make_et()
    et.fit(X_train, y_train)
    pred_et = et.predict(X_test)
    tau_et, _ = kendalltau(y_test, pred_et)
    et_hits, et_total = panel_top20_rate(y_test, pred_et, PANEL_IDX, test_i)

    # --- Stacking ensemble ---
    stk = make_stacker()
    stk.fit(X_train, y_train)
    pred_stk = stk.predict(X_test)
    tau_stk, _ = kendalltau(y_test, pred_stk)
    stk_hits, stk_total = panel_top20_rate(y_test, pred_stk, PANEL_IDX, test_i)

    # --- Simple average (no meta-learner) ---
    gb = make_gb()
    rf = make_rf()
    gb.fit(X_train, y_train)
    rf.fit(X_train, y_train)
    pred_avg = (pred_et + gb.predict(X_test) + rf.predict(X_test)) / 3.0
    tau_avg, _ = kendalltau(y_test, pred_avg)
    avg_hits, avg_total = panel_top20_rate(y_test, pred_avg, PANEL_IDX, test_i)

    elapsed = time.time() - t0
    results.append({
        "seed": seed,
        "tau_et": tau_et, "et_panel_hits": et_hits, "et_panel_total": et_total,
        "tau_stk": tau_stk, "stk_panel_hits": stk_hits, "stk_panel_total": stk_total,
        "tau_avg": tau_avg, "avg_panel_hits": avg_hits, "avg_panel_total": avg_total,
    })
    print(f"  Seed {seed}: ET={tau_et:.4f}  Stack={tau_stk:.4f}  Avg={tau_avg:.4f}  ({elapsed:.1f}s)")

res = pd.DataFrame(results)
print("\nAll 10 splits complete.")

  Seed 0: ET=0.2725  Stack=0.2660  Avg=0.2531  (5.8s)


  Seed 1: ET=0.3545  Stack=0.3573  Avg=0.3461  (5.1s)


  Seed 2: ET=0.2414  Stack=0.2247  Avg=0.2744  (4.8s)


  Seed 3: ET=0.2997  Stack=0.2920  Avg=0.3013  (4.5s)


  Seed 4: ET=0.3023  Stack=0.3006  Avg=0.2893  (4.7s)


  Seed 5: ET=0.3219  Stack=0.3176  Avg=0.3057  (4.4s)


  Seed 6: ET=0.3226  Stack=0.3149  Avg=0.3166  (4.4s)


  Seed 7: ET=0.3322  Stack=0.3294  Avg=0.3521  (4.4s)


  Seed 8: ET=0.2067  Stack=0.2060  Avg=0.1981  (4.4s)


  Seed 9: ET=0.3465  Stack=0.3414  Avg=0.3168  (4.7s)

All 10 splits complete.


In [4]:
print("=" * 72)
print("P-026  Ensemble Stacking Comparison  (10 splits, seeds 0-9)")
print("=" * 72)

mean_tau_et  = res["tau_et"].mean()
mean_tau_stk = res["tau_stk"].mean()
mean_tau_avg = res["tau_avg"].mean()

et_panel_rate  = res["et_panel_hits"].sum()  / max(res["et_panel_total"].sum(), 1)
stk_panel_rate = res["stk_panel_hits"].sum() / max(res["stk_panel_total"].sum(), 1)
avg_panel_rate = res["avg_panel_hits"].sum() / max(res["avg_panel_total"].sum(), 1)

print(f"\n{'Model':<22} {'Mean tau-b':>10} {'Panel top-20%':>14}")
print("-" * 48)
print(f"{'ET alone':<22} {mean_tau_et:>10.4f} {et_panel_rate:>13.1%}")
print(f"{'Stacking (Ridge)':<22} {mean_tau_stk:>10.4f} {stk_panel_rate:>13.1%}")
print(f"{'Simple average':<22} {mean_tau_avg:>10.4f} {avg_panel_rate:>13.1%}")

delta_stk = mean_tau_stk - mean_tau_et
delta_avg = mean_tau_avg - mean_tau_et

print(f"\nStacking vs ET:  delta tau-b = {delta_stk:+.4f}")
print(f"Average  vs ET:  delta tau-b = {delta_avg:+.4f}")

print("\nPer-split tau-b:")
for _, r in res.iterrows():
    s = int(r["seed"])
    print(f"  seed {s}: ET={r['tau_et']:.4f}  Stack={r['tau_stk']:.4f}  Avg={r['tau_avg']:.4f}"
          f"  | panel ET={int(r['et_panel_hits'])}/{int(r['et_panel_total'])}"
          f"  Stack={int(r['stk_panel_hits'])}/{int(r['stk_panel_total'])}"
          f"  Avg={int(r['avg_panel_hits'])}/{int(r['avg_panel_total'])}")

P-026  Ensemble Stacking Comparison  (10 splits, seeds 0-9)

Model                  Mean tau-b  Panel top-20%
------------------------------------------------
ET alone                   0.3000        100.0%
Stacking (Ridge)           0.2950        100.0%
Simple average             0.2954        100.0%

Stacking vs ET:  delta tau-b = -0.0050
Average  vs ET:  delta tau-b = -0.0047

Per-split tau-b:
  seed 0: ET=0.2725  Stack=0.2660  Avg=0.2531  | panel ET=0/0  Stack=0/0  Avg=0/0
  seed 1: ET=0.3545  Stack=0.3573  Avg=0.3461  | panel ET=1/1  Stack=1/1  Avg=1/1
  seed 2: ET=0.2414  Stack=0.2247  Avg=0.2744  | panel ET=1/1  Stack=1/1  Avg=1/1
  seed 3: ET=0.2997  Stack=0.2920  Avg=0.3013  | panel ET=0/0  Stack=0/0  Avg=0/0
  seed 4: ET=0.3023  Stack=0.3006  Avg=0.2893  | panel ET=0/0  Stack=0/0  Avg=0/0
  seed 5: ET=0.3219  Stack=0.3176  Avg=0.3057  | panel ET=1/1  Stack=1/1  Avg=1/1
  seed 6: ET=0.3226  Stack=0.3149  Avg=0.3166  | panel ET=0/0  Stack=0/0  Avg=0/0
  seed 7: ET=0.3322  Stack

In [5]:
out = res.copy()
out.to_csv("Packet_P026_Ensemble_Stacking.csv", index=False)
print("Saved Packet_P026_Ensemble_Stacking.csv")

Saved Packet_P026_Ensemble_Stacking.csv


In [6]:
# --- Status footer ---
panel_ok_stk = stk_panel_rate >= et_panel_rate - 0.05  # panel not materially worse

if delta_stk >= 0.01 and panel_ok_stk:
    status = "CONFIRMED"
    verdict = (f"Stacking improves tau-b by {delta_stk:+.4f} (>= +0.01) "
               f"with panel maintained. Stacking is beneficial.")
elif delta_stk >= 0.0 and panel_ok_stk:
    status = "PROMISING"
    verdict = (f"Stacking tau-b delta = {delta_stk:+.4f} (no worse than ET) "
               f"with panel maintained. Marginal or no gain.")
else:
    status = "NEGATIVE"
    if not panel_ok_stk:
        verdict = f"Stacking hurts panel consensus. Delta tau-b = {delta_stk:+.4f}."
    else:
        verdict = f"Stacking hurts tau-b (delta = {delta_stk:+.4f}). Not recommended."

print("=" * 72)
print(f"P-026 STATUS: {status}")
print(f"  {verdict}")
print("=" * 72)
print(f"\nMean tau-b:  ET={mean_tau_et:.4f}  Stack={mean_tau_stk:.4f}  Avg={mean_tau_avg:.4f}")
print(f"Panel top-20%: ET={et_panel_rate:.1%}  Stack={stk_panel_rate:.1%}  Avg={avg_panel_rate:.1%}")
print(f"\nSimple average delta vs ET: {delta_avg:+.4f}")
print(f"Stacking delta vs ET:      {delta_stk:+.4f}")

P-026 STATUS: NEGATIVE
  Stacking hurts tau-b (delta = -0.0050). Not recommended.

Mean tau-b:  ET=0.3000  Stack=0.2950  Avg=0.2954
Panel top-20%: ET=100.0%  Stack=100.0%  Avg=100.0%

Simple average delta vs ET: -0.0047
Stacking delta vs ET:      -0.0050
